<a href="https://colab.research.google.com/github/lanehale/pytorch-deep-learning/blob/main/07_tracking_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GitHub doesn't handle all output types. Text outputs have been included throughout, with links to images when necessary.

Optionally, see the full output in Colab notebook - https://colab.research.google.com/drive/1dXWXMuSja9YD8oiOgEC_r1PsOyec0MBn?usp=sharing

In [ ]:
# See if tensorboard exists, if not, install it
try:
  from torch.utils.tensorboard import SummaryWriter
  print("SummaryWriter already installed.")
except:
  print("Installing SummaryWriter...")
  !pip install -q tensorboard
  from torch.utils.tensorboard import SummaryWriter
  print("Done installing SummaryWriter.")



```
SummaryWriter already installed.
```



In [ ]:
"""
Create the going_modular folder and move in its scripts.
"""
import os

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
  from going_modular import data_setup, engine
  print("going_modular scripts already downloaded.")
except:
  # Get the going_modular scripts
  print("Downloading going_modular scripts...")
  !git clone https://github.com/lanehale/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  print("going_modular downloaded.")
  from going_modular import data_setup, engine

print(">!ls going_modular")
!ls going_modular



```
Downloading going_modular scripts...
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 380, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 380 (delta 100), reused 48 (delta 48), pack-reused 259 (from 2)
Receiving objects: 100% (380/380), 5.83 MiB | 18.82 MiB/s, done.
Resolving deltas: 100% (217/217), done.
going_modular downloaded.
>!ls going_modular
data_setup.py	  get_custom_data.py  pretrained_confmat.py  utils.py
download_data.py  get_data.py	      pretrained_writer.py
engine.py	  model_builder.py    __pycache__
get_any_data.py   predict.py	      train.py
```



In [ ]:
"""
Contains a function to run data through torchvision models and track the
results with SummaryWriter. (Saved in pretrained_writer.py for later)
"""
def run_model_writer(model,
                     weights,
                     train_dir,
                     test_dir,
                     batch_size,
                     dropout,
                     in_features,
                     optimizer_type,
                     optimizer_lr,
                     num_epochs,
                     image_data,
                     device,
                     writer,
                     model_name):

  auto_transforms = weights.transforms()

  # Create training and testing DataLoaders and get a list of class names
  train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
      train_dir=train_dir,
      test_dir=test_dir,
      transform=auto_transforms,  # perform the same data transforms on our training data as the pretrained model
      batch_size=batch_size
  )

  # Freeze all base layers in the "features" section of the model
  for parm in model.features.parameters():
    parm.requires_grad = False

  """ Adjust the output layer or the classifier portion of our pretrained model to our needs (out_features=3). """
  # Set manual seeds
  torch.manual_seed(42)
  torch.cuda.manual_seed(42)

  # Get the length of class_names (one output unit for each class)
  output_shape = len(class_names)

  # Recreate the classifier layer and seed it to the target device
  model.classifier = torch.nn.Sequential(
      torch.nn.Dropout(p=dropout, inplace=True),
      torch.nn.Linear(in_features=in_features,
                      out_features=output_shape,  # same number of output units as number of classes
                      bias=True)).to(device)

  # Define loss and optimizer
  loss_fn = nn.CrossEntropyLoss()
  if optimizer_type == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=optimizer_lr)
  else:
    optimizer = torch.optim.SGD(model.parameters(), lr=optimizer_lr)

  """ Train the model """
  # Start the timer
  from timeit import default_timer as timer
  start_time = timer()

  """ Note: We're only going to be training the parameters classifier here as all of the other parameters in our model have been frozen. """
  # Set up training and save the results
  print(f"Training the {model_name} model...")
  results = engine.train_writer(model=model,
                                train_dataloader=train_dataloader,
                                test_dataloader=test_dataloader,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                epochs=num_epochs,
                                device=device,
                                writer=writer)

  # End the timer and print out how long it took
  end_time = timer()
  print(f"[INFO] Total running time: {end_time - start_time:.3f} seconds")

  # Make predictions and store in a list of dictionaries
  print(f"Predicting with {test_dir} image_data...")
  pred_list, test_preds_tensor = predict_and_store(
      model=model,
      test_paths=image_data,
      tranform=auto_transforms,
      class_names=class_names,
      device=device
  )

  print(f"Max test acc: {max(results['test_acc']):.3f} | Min test loss: {min(results['test_loss']):.3f}")

  return results, pred_list

In [ ]:
# Download 10 percent and 20 percent training data
from going_modular import download_data

data_10_percent_path = download_data.from_path(from_path="pizza_steak_sushi.zip", image_dir="pizza_steak_sushi")
data_20_percent_path = download_data.from_path(from_path="pizza_steak_sushi_20_percent.zip", image_dir="pizza_steak_sushi_20_percent")



```
Did not find data/pizza_steak_sushi directory, creating one...
Downloading pizza_steak_sushi data...
Unzipping pizza_steak_sushi data...
>!ls data/pizza_steak_sushi
test
train

Did not find data/pizza_steak_sushi_20_percent directory, creating one...
Downloading pizza_steak_sushi_20_percent data...
Unzipping pizza_steak_sushi_20_percent data...
>!ls data/pizza_steak_sushi_20_percent
test
train
```



In [ ]:
# Set up training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Setup testing directory paths ('his' note: use the same test dataset for both to compare the results)
"""
I disagree with his note above, but we will compare it with using 20 percent test data later.
Using the same test_dir on all models would be: 10% dataset train/test = 67/33 (150/225, 75/225)
but 20% dataset train/test = 67/17 (300/450, 75/450) with unused test data, or 300/375, 75/375 = 80/20.
So using the same test dataset for both wouldn't compare apples to apples. Also, his test_dataloader shows
a length of 8 which doesn't match the length of the 10% test dataset divided by batch size (75/32 = 3),
which implies he used 225 images in the test dataset. Did he combine the 10% and 20% test datasets?
"""
test_dir_10 = data_10_percent_path / "test"
test_dir_20 = data_20_percent_path / "test"

train_dir_10_percent, train_dir_20_percent, test_dir_10, test_dir_20



```
(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi_20_percent/train'),
 PosixPath('data/pizza_steak_sushi/test'),
 PosixPath('data/pizza_steak_sushi_20_percent/test'))
```



In [ ]:
import torch
import torchvision

from pathlib import Path

test_image_path_list_10 = list(Path(test_dir_10).glob("*/*.jpg"))  # this is only used for predictions
test_image_path_list_20 = list(Path(test_dir_20).glob("*/*.jpg"))  # this is only used for predictions

weights_b0 = torchvision.models.EfficientNet_B0_Weights.DEFAULT  # .DEFAULT = best available weights from pretraining on ImageNet
weights_b2 = torchvision.models.EfficientNet_B2_Weights.DEFAULT

# Create models list (need to create a new model for each experiment)
model_parameters = {"effnetb0": {"weights": weights_b0, "in_features": 1280, "dropout": 0.2},
                    "effnetb2": {"weights": weights_b2, "in_features": 1408, "dropout": 0.3},
                    }

# Create data paths list
data_paths = [["data_10_percent", train_dir_10_percent, test_dir_10, test_image_path_list_10],
              ["data_20_percent", train_dir_20_percent, test_dir_20, test_image_path_list_20],
              ["train_20_test_10_percent", train_dir_20_percent, test_dir_10, test_image_path_list_10],
              ]

# Create epochs list
num_epochs = [5, 10]

print(test_image_path_list_10[:3])
print(test_image_path_list_10[37:40])
print(test_image_path_list_10[-3:])
print()
print(test_image_path_list_20[:3])
print(test_image_path_list_20[75:78])
print(test_image_path_list_20[-3:])



```
[PosixPath('data/pizza_steak_sushi/test/steak/27415.jpg'), PosixPath('data/pizza_steak_sushi/test/steak/100274.jpg'), PosixPath('data/pizza_steak_sushi/test/steak/673127.jpg')]
[PosixPath('data/pizza_steak_sushi/test/sushi/1172255.jpg'), PosixPath('data/pizza_steak_sushi/test/sushi/3806282.jpg'), PosixPath('data/pizza_steak_sushi/test/sushi/2521706.jpg')]
[PosixPath('data/pizza_steak_sushi/test/pizza/2508636.jpg'), PosixPath('data/pizza_steak_sushi/test/pizza/540882.jpg'), PosixPath('data/pizza_steak_sushi/test/pizza/3475871.jpg')]

[PosixPath('data/pizza_steak_sushi_20_percent/test/steak/552171.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/steak/40947.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/steak/2061929.jpg')]
[PosixPath('data/pizza_steak_sushi_20_percent/test/sushi/1346344.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/sushi/720302.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/sushi/715227.jpg')]
[PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/138961.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/2549661.jpg'), PosixPath('data/pizza_steak_sushi_20_percent/test/pizza/1315645.jpg')]
```



In [ ]:
%%time
"""
Run experiments
"""
from torch import nn
from going_modular import pretrained_writer as pretrained
from going_modular.utils import create_writer, save_model

# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
optimizer_type = "Adam"
optimizer_lr = 0.001

# Keep track of experiment numbers
experiment_number = 0

completed_experiments = {}

for model_name in model_parameters:
  weights = model_parameters[model_name]["weights"]
  in_features = model_parameters[model_name]["in_features"]
  dropout = model_parameters[model_name]["dropout"]

  for data in data_paths:
    dataset_name = data[0]
    train_dir = data[1]
    test_dir = data[2]
    image_data = data[3]

    for epochs in num_epochs:
      model_to_train = f"{experiment_number}_{model_name}_{dataset_name}_{epochs}_epochs"
      model_to_train_str = model_to_train
      results = "results_" + model_to_train
      predictions = "predictions_" + model_to_train

      if model_name == "effnetb0":
        model_to_train = torchvision.models.efficientnet_b0(weights=weights_b0).to(device)
      elif model_name == "effnetb2":
        model_to_train = torchvision.models.efficientnet_b2(weights=weights_b2).to(device)

      results, predictions = pretrained.run_model_writer(
          model=model_to_train,
          weights=weights,
          train_dir=train_dir,
          test_dir=test_dir,
          batch_size=BATCH_SIZE,
          dropout=dropout,
          in_features=in_features,
          optimizer_type=optimizer_type,
          optimizer_lr=optimizer_lr,
          num_epochs=epochs,
          image_data=image_data,
          device=device,
          writer=create_writer(experiment_name=dataset_name,
                               model_name=model_name,
                               extra=f"{epochs}_epochs"),
          model_name=model_to_train_str
      )

      # Save experiment info for later
      completed_experiments[experiment_number] = [model_to_train_str, model_to_train, results, predictions]

      experiment_number += 1

      # Save the model to file so we can get back the best model
      save_model(model=model_to_train,
                 target_dir="models",
                 model_name=model_to_train_str + ".pth")
      print("-"*50 + "\n")



```
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 189MB/s]
[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_10_percent/effnetb0/5_epochs...
Training with model 0_effnetb0_data_10_percent_5_epochs...
100%
 5/5 [00:13<00:00,  2.55s/it]
Epoch: 1 | train_loss: 1.0313 | train_acc: 0.5117 | test_loss: 0.8690 | test_acc: 0.6723
Epoch: 2 | train_loss: 0.8755 | train_acc: 0.6953 | test_loss: 0.7995 | test_acc: 0.7027
Epoch: 3 | train_loss: 0.7953 | train_acc: 0.6523 | test_loss: 0.6977 | test_acc: 0.8248
Epoch: 4 | train_loss: 0.7233 | train_acc: 0.7656 | test_loss: 0.5791 | test_acc: 0.9062
Epoch: 5 | train_loss: 0.6085 | train_acc: 0.8906 | test_loss: 0.5739 | test_acc: 0.8759
[INFO] Total running time: 13.151 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.906 | Min test loss: 0.574
[INFO] Saving model to: models/0_effnetb0_data_10_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_10_percent/effnetb0/10_epochs...
Training with model 1_effnetb0_data_10_percent_10_epochs...
100%
 10/10 [00:24<00:00,  2.47s/it]
Epoch: 1 | train_loss: 1.0313 | train_acc: 0.5117 | test_loss: 0.8690 | test_acc: 0.6723
Epoch: 2 | train_loss: 0.8755 | train_acc: 0.6953 | test_loss: 0.7995 | test_acc: 0.7027
Epoch: 3 | train_loss: 0.7953 | train_acc: 0.6523 | test_loss: 0.6977 | test_acc: 0.8248
Epoch: 4 | train_loss: 0.7233 | train_acc: 0.7656 | test_loss: 0.5791 | test_acc: 0.9062
Epoch: 5 | train_loss: 0.6085 | train_acc: 0.8906 | test_loss: 0.5739 | test_acc: 0.8759
Epoch: 6 | train_loss: 0.5446 | train_acc: 0.9375 | test_loss: 0.5947 | test_acc: 0.8456
Epoch: 7 | train_loss: 0.5724 | train_acc: 0.7891 | test_loss: 0.5502 | test_acc: 0.8352
Epoch: 8 | train_loss: 0.4594 | train_acc: 0.9531 | test_loss: 0.4931 | test_acc: 0.8561
Epoch: 9 | train_loss: 0.5643 | train_acc: 0.7891 | test_loss: 0.4967 | test_acc: 0.8456
Epoch: 10 | train_loss: 0.4993 | train_acc: 0.7969 | test_loss: 0.4784 | test_acc: 0.8561
[INFO] Total running time: 24.012 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.906 | Min test loss: 0.478
[INFO] Saving model to: models/1_effnetb0_data_10_percent_10_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_20_percent/effnetb0/5_epochs...
Training with model 2_effnetb0_data_20_percent_5_epochs...
100%
 5/5 [00:24<00:00,  4.81s/it]
Epoch: 1 | train_loss: 0.9514 | train_acc: 0.5979 | test_loss: 0.6660 | test_acc: 0.8568
Epoch: 2 | train_loss: 0.6948 | train_acc: 0.7896 | test_loss: 0.5529 | test_acc: 0.8670
Epoch: 3 | train_loss: 0.5757 | train_acc: 0.8354 | test_loss: 0.4710 | test_acc: 0.8818
Epoch: 4 | train_loss: 0.5072 | train_acc: 0.8417 | test_loss: 0.4227 | test_acc: 0.8881
Epoch: 5 | train_loss: 0.4902 | train_acc: 0.8063 | test_loss: 0.3990 | test_acc: 0.8881
[INFO] Total running time: 24.479 seconds
Predicting with data/pizza_steak_sushi_20_percent/test image_data...
Max test acc: 0.888 | Min test loss: 0.399
[INFO] Saving model to: models/2_effnetb0_data_20_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_20_percent/effnetb0/10_epochs...
Training with model 3_effnetb0_data_20_percent_10_epochs...
100%
 10/10 [00:46<00:00,  4.75s/it]
Epoch: 1 | train_loss: 0.9514 | train_acc: 0.5979 | test_loss: 0.6660 | test_acc: 0.8568
Epoch: 2 | train_loss: 0.6948 | train_acc: 0.7896 | test_loss: 0.5529 | test_acc: 0.8670
Epoch: 3 | train_loss: 0.5757 | train_acc: 0.8354 | test_loss: 0.4710 | test_acc: 0.8818
Epoch: 4 | train_loss: 0.5072 | train_acc: 0.8417 | test_loss: 0.4227 | test_acc: 0.8881
Epoch: 5 | train_loss: 0.4902 | train_acc: 0.8063 | test_loss: 0.3990 | test_acc: 0.8881
Epoch: 6 | train_loss: 0.3797 | train_acc: 0.9187 | test_loss: 0.3863 | test_acc: 0.9006
Epoch: 7 | train_loss: 0.3710 | train_acc: 0.9104 | test_loss: 0.3517 | test_acc: 0.9097
Epoch: 8 | train_loss: 0.3500 | train_acc: 0.8917 | test_loss: 0.3306 | test_acc: 0.9102
Epoch: 9 | train_loss: 0.2974 | train_acc: 0.9187 | test_loss: 0.3117 | test_acc: 0.9006
Epoch: 10 | train_loss: 0.3665 | train_acc: 0.8771 | test_loss: 0.3169 | test_acc: 0.8943
[INFO] Total running time: 46.502 seconds
Predicting with data/pizza_steak_sushi_20_percent/test image_data...
Max test acc: 0.910 | Min test loss: 0.312
[INFO] Saving model to: models/3_effnetb0_data_20_percent_10_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/train_20_test_10_percent/effnetb0/5_epochs...
Training with model 4_effnetb0_train_20_test_10_percent_5_epochs...
100%
 5/5 [00:20<00:00,  3.98s/it]
Epoch: 1 | train_loss: 0.9514 | train_acc: 0.5979 | test_loss: 0.6642 | test_acc: 0.9271
Epoch: 2 | train_loss: 0.6948 | train_acc: 0.7896 | test_loss: 0.5717 | test_acc: 0.9479
Epoch: 3 | train_loss: 0.5757 | train_acc: 0.8354 | test_loss: 0.4476 | test_acc: 0.9271
Epoch: 4 | train_loss: 0.5072 | train_acc: 0.8417 | test_loss: 0.4159 | test_acc: 0.9489
Epoch: 5 | train_loss: 0.4902 | train_acc: 0.8063 | test_loss: 0.3911 | test_acc: 0.9280
[INFO] Total running time: 20.112 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.949 | Min test loss: 0.391
[INFO] Saving model to: models/4_effnetb0_train_20_test_10_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/train_20_test_10_percent/effnetb0/10_epochs...
Training with model 5_effnetb0_train_20_test_10_percent_10_epochs...
100%
 10/10 [00:40<00:00,  4.14s/it]
Epoch: 1 | train_loss: 0.9514 | train_acc: 0.5979 | test_loss: 0.6642 | test_acc: 0.9271
Epoch: 2 | train_loss: 0.6948 | train_acc: 0.7896 | test_loss: 0.5717 | test_acc: 0.9479
Epoch: 3 | train_loss: 0.5757 | train_acc: 0.8354 | test_loss: 0.4476 | test_acc: 0.9271
Epoch: 4 | train_loss: 0.5072 | train_acc: 0.8417 | test_loss: 0.4159 | test_acc: 0.9489
Epoch: 5 | train_loss: 0.4902 | train_acc: 0.8063 | test_loss: 0.3911 | test_acc: 0.9280
Epoch: 6 | train_loss: 0.3797 | train_acc: 0.9187 | test_loss: 0.3706 | test_acc: 0.9479
Epoch: 7 | train_loss: 0.3710 | train_acc: 0.9104 | test_loss: 0.3187 | test_acc: 0.9479
Epoch: 8 | train_loss: 0.3500 | train_acc: 0.8917 | test_loss: 0.3433 | test_acc: 0.8977
Epoch: 9 | train_loss: 0.2974 | train_acc: 0.9187 | test_loss: 0.3193 | test_acc: 0.9176
Epoch: 10 | train_loss: 0.3665 | train_acc: 0.8771 | test_loss: 0.3001 | test_acc: 0.9072
[INFO] Total running time: 40.217 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.949 | Min test loss: 0.300
[INFO] Saving model to: models/5_effnetb0_train_20_test_10_percent_10_epochs.pth
--------------------------------------------------

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth
100%|██████████| 35.2M/35.2M [00:00<00:00, 130MB/s]
[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_10_percent/effnetb2/5_epochs...
Training with model 6_effnetb2_data_10_percent_5_epochs...
100%
 5/5 [00:16<00:00,  3.17s/it]
Epoch: 1 | train_loss: 1.0852 | train_acc: 0.3867 | test_loss: 0.9224 | test_acc: 0.7538
Epoch: 2 | train_loss: 0.9143 | train_acc: 0.6641 | test_loss: 0.8252 | test_acc: 0.8059
Epoch: 3 | train_loss: 0.8217 | train_acc: 0.7539 | test_loss: 0.6787 | test_acc: 0.8968
Epoch: 4 | train_loss: 0.6747 | train_acc: 0.9023 | test_loss: 0.6456 | test_acc: 0.8665
Epoch: 5 | train_loss: 0.6669 | train_acc: 0.8242 | test_loss: 0.6390 | test_acc: 0.8466
[INFO] Total running time: 16.195 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.897 | Min test loss: 0.639
[INFO] Saving model to: models/6_effnetb2_data_10_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_10_percent/effnetb2/10_epochs...
Training with model 7_effnetb2_data_10_percent_10_epochs...
100%
 10/10 [00:31<00:00,  3.22s/it]
Epoch: 1 | train_loss: 1.0852 | train_acc: 0.3867 | test_loss: 0.9224 | test_acc: 0.7538
Epoch: 2 | train_loss: 0.9143 | train_acc: 0.6641 | test_loss: 0.8252 | test_acc: 0.8059
Epoch: 3 | train_loss: 0.8217 | train_acc: 0.7539 | test_loss: 0.6787 | test_acc: 0.8968
Epoch: 4 | train_loss: 0.6747 | train_acc: 0.9023 | test_loss: 0.6456 | test_acc: 0.8665
Epoch: 5 | train_loss: 0.6669 | train_acc: 0.8242 | test_loss: 0.6390 | test_acc: 0.8466
Epoch: 6 | train_loss: 0.5795 | train_acc: 0.8008 | test_loss: 0.5625 | test_acc: 0.9072
Epoch: 7 | train_loss: 0.5668 | train_acc: 0.8125 | test_loss: 0.5567 | test_acc: 0.9280
Epoch: 8 | train_loss: 0.5009 | train_acc: 0.9414 | test_loss: 0.5881 | test_acc: 0.8674
Epoch: 9 | train_loss: 0.5177 | train_acc: 0.8047 | test_loss: 0.5837 | test_acc: 0.8570
Epoch: 10 | train_loss: 0.4651 | train_acc: 0.9453 | test_loss: 0.5410 | test_acc: 0.8570
[INFO] Total running time: 31.944 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.928 | Min test loss: 0.541
[INFO] Saving model to: models/7_effnetb2_data_10_percent_10_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_20_percent/effnetb2/5_epochs...
Training with model 8_effnetb2_data_20_percent_5_epochs...
100%
 5/5 [00:28<00:00,  5.73s/it]
Epoch: 1 | train_loss: 0.9791 | train_acc: 0.5604 | test_loss: 0.7128 | test_acc: 0.9011
Epoch: 2 | train_loss: 0.7206 | train_acc: 0.8063 | test_loss: 0.5769 | test_acc: 0.9443
Epoch: 3 | train_loss: 0.5971 | train_acc: 0.7937 | test_loss: 0.4906 | test_acc: 0.9437
Epoch: 4 | train_loss: 0.5227 | train_acc: 0.8271 | test_loss: 0.4484 | test_acc: 0.9222
Epoch: 5 | train_loss: 0.4196 | train_acc: 0.8917 | test_loss: 0.3825 | test_acc: 0.9375
[INFO] Total running time: 28.825 seconds
Predicting with data/pizza_steak_sushi_20_percent/test image_data...
Max test acc: 0.944 | Min test loss: 0.382
[INFO] Saving model to: models/8_effnetb2_data_20_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/data_20_percent/effnetb2/10_epochs...
Training with model 9_effnetb2_data_20_percent_10_epochs...
100%
 10/10 [00:57<00:00,  5.65s/it]
Epoch: 1 | train_loss: 0.9791 | train_acc: 0.5604 | test_loss: 0.7128 | test_acc: 0.9011
Epoch: 2 | train_loss: 0.7206 | train_acc: 0.8063 | test_loss: 0.5769 | test_acc: 0.9443
Epoch: 3 | train_loss: 0.5971 | train_acc: 0.7937 | test_loss: 0.4906 | test_acc: 0.9437
Epoch: 4 | train_loss: 0.5227 | train_acc: 0.8271 | test_loss: 0.4484 | test_acc: 0.9222
Epoch: 5 | train_loss: 0.4196 | train_acc: 0.8917 | test_loss: 0.3825 | test_acc: 0.9375
Epoch: 6 | train_loss: 0.3838 | train_acc: 0.9083 | test_loss: 0.3493 | test_acc: 0.9443
Epoch: 7 | train_loss: 0.3517 | train_acc: 0.9208 | test_loss: 0.3165 | test_acc: 0.9688
Epoch: 8 | train_loss: 0.3706 | train_acc: 0.9062 | test_loss: 0.3061 | test_acc: 0.9534
Epoch: 9 | train_loss: 0.3070 | train_acc: 0.9396 | test_loss: 0.2988 | test_acc: 0.9597
Epoch: 10 | train_loss: 0.3613 | train_acc: 0.8958 | test_loss: 0.2692 | test_acc: 0.9750
[INFO] Total running time: 57.202 seconds
Predicting with data/pizza_steak_sushi_20_percent/test image_data...
Max test acc: 0.975 | Min test loss: 0.269
[INFO] Saving model to: models/9_effnetb2_data_20_percent_10_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/train_20_test_10_percent/effnetb2/5_epochs...
Training with model 10_effnetb2_train_20_test_10_percent_5_epochs...
100%
 5/5 [00:25<00:00,  4.98s/it]
Epoch: 1 | train_loss: 0.9791 | train_acc: 0.5604 | test_loss: 0.7433 | test_acc: 0.8864
Epoch: 2 | train_loss: 0.7206 | train_acc: 0.8063 | test_loss: 0.6122 | test_acc: 0.8873
Epoch: 3 | train_loss: 0.5971 | train_acc: 0.7937 | test_loss: 0.5005 | test_acc: 0.9072
Epoch: 4 | train_loss: 0.5227 | train_acc: 0.8271 | test_loss: 0.5143 | test_acc: 0.8873
Epoch: 5 | train_loss: 0.4196 | train_acc: 0.8917 | test_loss: 0.3956 | test_acc: 0.9176
[INFO] Total running time: 25.563 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.918 | Min test loss: 0.396
[INFO] Saving model to: models/10_effnetb2_train_20_test_10_percent_5_epochs.pth
--------------------------------------------------

[INFO] Created SummaryWriter, saving to: runs/2025-05-31/train_20_test_10_percent/effnetb2/10_epochs...
Training with model 11_effnetb2_train_20_test_10_percent_10_epochs...
100%
 10/10 [00:51<00:00,  5.05s/it]
Epoch: 1 | train_loss: 0.9791 | train_acc: 0.5604 | test_loss: 0.7433 | test_acc: 0.8864
Epoch: 2 | train_loss: 0.7206 | train_acc: 0.8063 | test_loss: 0.6122 | test_acc: 0.8873
Epoch: 3 | train_loss: 0.5971 | train_acc: 0.7937 | test_loss: 0.5005 | test_acc: 0.9072
Epoch: 4 | train_loss: 0.5227 | train_acc: 0.8271 | test_loss: 0.5143 | test_acc: 0.8873
Epoch: 5 | train_loss: 0.4196 | train_acc: 0.8917 | test_loss: 0.3956 | test_acc: 0.9176
Epoch: 6 | train_loss: 0.3838 | train_acc: 0.9083 | test_loss: 0.4087 | test_acc: 0.8873
Epoch: 7 | train_loss: 0.3517 | train_acc: 0.9208 | test_loss: 0.3589 | test_acc: 0.9280
Epoch: 8 | train_loss: 0.3706 | train_acc: 0.9062 | test_loss: 0.3690 | test_acc: 0.8873
Epoch: 9 | train_loss: 0.3070 | train_acc: 0.9396 | test_loss: 0.3627 | test_acc: 0.8873
Epoch: 10 | train_loss: 0.3613 | train_acc: 0.8958 | test_loss: 0.3455 | test_acc: 0.8873
[INFO] Total running time: 51.234 seconds
Predicting with data/pizza_steak_sushi/test image_data...
Max test acc: 0.928 | Min test loss: 0.346
[INFO] Saving model to: models/11_effnetb2_train_20_test_10_percent_10_epochs.pth
--------------------------------------------------

CPU times: user 1min 46s, sys: 20.7 s, total: 2min 7s
Wall time: 6min 48s
```



In [ ]:
print(f"{len(completed_experiments)} experiments completed:")
!ls models



```
12 experiments completed:
0_effnetb0_data_10_percent_5_epochs.pth
10_effnetb2_train_20_test_10_percent_5_epochs.pth
11_effnetb2_train_20_test_10_percent_10_epochs.pth
1_effnetb0_data_10_percent_10_epochs.pth
2_effnetb0_data_20_percent_5_epochs.pth
3_effnetb0_data_20_percent_10_epochs.pth
4_effnetb0_train_20_test_10_percent_5_epochs.pth
5_effnetb0_train_20_test_10_percent_10_epochs.pth
6_effnetb2_data_10_percent_5_epochs.pth
7_effnetb2_data_10_percent_10_epochs.pth
8_effnetb2_data_20_percent_5_epochs.pth
9_effnetb2_data_20_percent_10_epochs.pth
```



In [ ]:
# Create a function to display results (Saved in utils.py for later)
def compare_results(pred_list, name):
  false_count = 0
  for pred in pred_list:
    if pred['correct'] == False:
      false_count += 1
  false_percent = 100 * false_count / len(pred_list)
  print(
      f"{name :<46} | False predictions: {false_count :<2} out of {len(pred_list) :<3}, "
      f"or {false_percent:5.2f}% wrong, "
      f"{(100.0 - false_percent):.2f}% right"
  )

In [ ]:
for experiment in completed_experiments:
  predictions = completed_experiments[experiment][3]
  name = completed_experiments[experiment][0]
  compare_results(predictions, name)



```
0_effnetb0_data_10_percent_5_epochs            | False predictions: 10 out of 75 , or 13.33% wrong, 86.67% right
1_effnetb0_data_10_percent_10_epochs           | False predictions: 10 out of 75 , or 13.33% wrong, 86.67% right
2_effnetb0_data_20_percent_5_epochs            | False predictions: 17 out of 150, or 11.33% wrong, 88.67% right
3_effnetb0_data_20_percent_10_epochs           | False predictions: 16 out of 150, or 10.67% wrong, 89.33% right
4_effnetb0_train_20_test_10_percent_5_epochs   | False predictions: 5  out of 75 , or  6.67% wrong, 93.33% right
5_effnetb0_train_20_test_10_percent_10_epochs  | False predictions: 7  out of 75 , or  9.33% wrong, 90.67% right
6_effnetb2_data_10_percent_5_epochs            | False predictions: 9  out of 75 , or 12.00% wrong, 88.00% right
7_effnetb2_data_10_percent_10_epochs           | False predictions: 8  out of 75 , or 10.67% wrong, 89.33% right
8_effnetb2_data_20_percent_5_epochs            | False predictions: 10 out of 150, or  6.67% wrong, 93.33% right
9_effnetb2_data_20_percent_10_epochs           | False predictions: 4  out of 150, or  2.67% wrong, 97.33% right
10_effnetb2_train_20_test_10_percent_5_epochs  | False predictions: 6  out of 75 , or  8.00% wrong, 92.00% right
11_effnetb2_train_20_test_10_percent_10_epochs | False predictions: 7  out of 75 , or  9.33% wrong, 90.67% right
```



In [ ]:
index = 0
best_model_index = 0
best_model_acc = 0.0

for experiment in completed_experiments:
  results = completed_experiments[experiment][2]
  max_test_acc = max(results['test_acc'])
  index += 1

  if max_test_acc > best_model_acc:
    best_model_acc = max_test_acc
    best_model_index = index

  print(
      f"{completed_experiments[experiment][0] :<46} | "
      f"Max test acc: {max_test_acc:.3f} | "
      f"Min test loss: {min(results['test_loss']):.3f}"
  )



```
0_effnetb0_data_10_percent_5_epochs            | Max test acc: 0.906 | Min test loss: 0.574
1_effnetb0_data_10_percent_10_epochs           | Max test acc: 0.906 | Min test loss: 0.478
2_effnetb0_data_20_percent_5_epochs            | Max test acc: 0.888 | Min test loss: 0.399
3_effnetb0_data_20_percent_10_epochs           | Max test acc: 0.910 | Min test loss: 0.312
4_effnetb0_train_20_test_10_percent_5_epochs   | Max test acc: 0.949 | Min test loss: 0.391
5_effnetb0_train_20_test_10_percent_10_epochs  | Max test acc: 0.949 | Min test loss: 0.300
6_effnetb2_data_10_percent_5_epochs            | Max test acc: 0.897 | Min test loss: 0.639
7_effnetb2_data_10_percent_10_epochs           | Max test acc: 0.928 | Min test loss: 0.541
8_effnetb2_data_20_percent_5_epochs            | Max test acc: 0.944 | Min test loss: 0.382
9_effnetb2_data_20_percent_10_epochs           | Max test acc: 0.975 | Min test loss: 0.269
10_effnetb2_train_20_test_10_percent_5_epochs  | Max test acc: 0.918 | Min test loss: 0.396
11_effnetb2_train_20_test_10_percent_10_epochs | Max test acc: 0.928 | Min test loss: 0.346
```



In [ ]:
# Save the models locally to my machine
from google.colab import files

for experiment in completed_experiments:
  model_name = completed_experiments[experiment][0]
  model_path = "models/" + model_name + ".pth"  # gets model from here and
  files.download(model_path)                    # saves it to my Downloads folder

In [ ]:
# View TensorBoard (in Jupyter and Colab Notebooks)
%load_ext tensorboard
%tensorboard --logdir runs

Image Links:

TensorBoard accuracy - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/TensorBoard_accuracy.png

TensorBoard accuracy details - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/TensorBoard_accuracy_detailsUntitled.png

TensorBoard loss - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/TensorBoard_loss.png

TensorBoard loss details - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/TensorBoard_loss_details.png

In [ ]:
# Predict on a random sample of images
from going_modular.utils import pred_and_plot_image

best_model = completed_experiments[best_model_index][1]
class_names = ["pizza", "steak", "sushi"]

# Get a random list of 5 images from 20% test set
import random
random_image_paths = random.sample(test_image_path_list_20, k=5)

# Iterate through random test image paths, make predictions on them and plot them
for image_path in random_image_paths:
  pred_and_plot_image(model=best_model,
                      image_path=image_path,
                      class_names=class_names,
                      extra_Title=True,
                      # Notice bad predictions without transform= (need to transform the image to predict correctly)
                      image_size=(224, 224))

Image Links:

predicted sushi on steak - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_sushi_on_steak.png

predicted steak on sushi - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_steak_on_sushi.png

predicted sushi on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_sushi_on_pizza.png

predicted pizza on sushi - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_pizza_on_sushi.png

In [ ]:
"""
Predict on my own images
"""
# Get custom images
!python going_modular/get_custom_data.py



```
DownLoading data/cheese-pizza.jpeg...
DownLoading data/pizza-slice.jpeg...
DownLoading data/pizza-slice2.jpeg...
DownLoading data/pizza-sliced.jpeg...
DownLoading data/pizza-sliced2.jpeg...
DownLoading data/pizza-partial-view.jpeg...
DownLoading data/pizza-partial-view2.jpeg...
DownLoading data/pizza-side-view.jpeg...
```



In [ ]:
data_path = Path("data")

filenames = [
    "cheese-pizza.jpeg",
    "pizza-slice.jpeg",
    "pizza-slice2.jpeg",
    "pizza-sliced.jpeg",
    "pizza-sliced2.jpeg",
    "pizza-partial-view.jpeg",
    "pizza-partial-view2.jpeg",
    "pizza-side-view.jpeg"
]

for f in filenames:
  # Set custom image path
  custom_image_path = data_path / f
  # Predict on custom image
  pred_and_plot_image(model=best_model,
                      image_path=custom_image_path,
                      class_names=class_names,
                      # Notice bad predictions without transform= (need to transform the image to correctly predict)
                      image_size=(224, 224))

Image Links:

predicted steak on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_custom_image_2.png

predicted sushi on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_custom_image_3.png

predicted steak on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_custom_image_4.png

predicted sushi on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_custom_image_6.png

predicted steak on pizza - https://github.com/lanehale/pytorch-deep-learning/blob/main/07-output-images/bad_prediction_custom_image_8.png